
# KAMARA — NLP
## Stress test de shortcuts sintéticos no BERTimbau

**Experimento:** `kamara_bertimbau_v1`  
**Objetivo:** verificar se o desempenho do BERTimbau na validation depende
fortemente de frases/expressões associadas ao processo de geração sintética.

Este notebook:

- **não treina novamente** o modelo;
- usa apenas `validation.jsonl`;
- **não usa `test.jsonl`**;
- carrega o modelo já treinado em `kamara_bertimbau_v1_export.zip`;
- mede o desempenho original;
- mascara expressões suspeitas;
- mede novamente;
- registra prediction flips, confiança e métricas por classe.

A interpretação correta é:

- queda pequena → baixa dependência das expressões avaliadas;
- queda grande → possível dependência de assinatura sintética;
- mesmo com queda pequena, isso **não prova ausência de todo viés sintético**.



## 0. Arquivos necessários

Faça upload somente de:

```text
validation.jsonl
kamara_bertimbau_v1_export.zip
```

Não envie `test.jsonl`.


In [1]:

# 1. Ambiente
import os
import platform
import subprocess
import sys

print("Python:", sys.version)
print("Plataforma:", platform.platform())

try:
    print(subprocess.check_output(["nvidia-smi"], text=True))
except Exception as exc:
    print("nvidia-smi indisponível:", repr(exc))


Python: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
Plataforma: Linux-6.6.122+-x86_64-with-glibc2.35
Sun Aug 30 14:30:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                               

In [2]:

# 2. Dependências
!pip -q install -U \
    "transformers>=4.50,<6" \
    "scikit-learn>=1.5,<2" \
    "pandas>=2,<4" \
    "safetensors>=0.4,<1"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 67.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.5 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.5 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.5 which is incompatible.


In [3]:

# 3. Imports
import csv
import hashlib
import json
import re
import shutil
import zipfile
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import torch
import transformers

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from transformers import AutoModelForSequenceClassification, AutoTokenizer

print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)
print("sklearn     :", sklearn.__version__)
print("CUDA        :", torch.cuda.is_available())

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE      :", DEVICE)

if torch.cuda.is_available():
    print("GPU         :", torch.cuda.get_device_name(0))


torch       : 2.11.0+cu128
transformers: 5.16.1
sklearn     : 1.9.0
CUDA        : True
DEVICE      : cuda
GPU         : Tesla T4


In [6]:
from pathlib import Path

CONTENT = Path("/content")

validation_candidates = sorted(
    CONTENT.glob("validation*.jsonl"),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)

zip_candidates = sorted(
    CONTENT.glob("kamara_bertimbau_v1_export*.zip"),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)

if not validation_candidates:
    raise FileNotFoundError("Nenhum validation*.jsonl encontrado em /content")

if not zip_candidates:
    raise FileNotFoundError(
        "Nenhum kamara_bertimbau_v1_export*.zip encontrado em /content"
    )

# Usa os arquivos enviados mais recentemente
VALIDATION_PATH = validation_candidates[0]
ZIP_PATH = zip_candidates[0]

print("Usando arquivos já existentes:")
print("Validation:", VALIDATION_PATH)
print("ZIP       :", ZIP_PATH)

print("\nNenhum novo upload necessário.")
print("TESTE NÃO FOI UTILIZADO.")

Usando arquivos já existentes:
Validation: /content/validation (1).jsonl
ZIP       : /content/kamara_bertimbau_v1_export (1).zip

Nenhum novo upload necessário.
TESTE NÃO FOI UTILIZADO.


In [7]:
# 5. Extrair modelo exportado
EXTRACT_ROOT = Path("/content/kamara_bertimbau_export")

if EXTRACT_ROOT.exists():
    shutil.rmtree(EXTRACT_ROOT)

EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zf.extractall(EXTRACT_ROOT)

MODEL_DIR = (
    EXTRACT_ROOT
    / "ml"
    / "artifacts"
    / "nlp"
    / "kamara_bertimbau_v1"
)

if not MODEL_DIR.is_dir():
    raise FileNotFoundError(
        f"Diretório do modelo não encontrado: {MODEL_DIR}"
    )

print("Modelo encontrado em:", MODEL_DIR)

for path in sorted(MODEL_DIR.iterdir()):
    if path.is_file():
        print(" -", path.name)

Modelo encontrado em: /content/kamara_bertimbau_export/ml/artifacts/nlp/kamara_bertimbau_v1
 - config.json
 - model.safetensors
 - run_metadata.json
 - tokenizer.json
 - tokenizer_config.json
 - training_args.bin


In [8]:

# 6. Carregar validation e conferir integridade

EXPECTED_LABELS = [
    "COBRADE_1_2_1_0_0",
    "COBRADE_1_2_2_0_0",
    "COBRADE_1_2_3_0_0",
    "COBRADE_1_4_1_3_1",
    "COBRADE_1_4_1_3_2",
    "COBRADE_2_3_1_1_0",
    "COBRADE_2_3_1_2_0",
    "OTHER",
    "INSUFFICIENT_INFO",
]

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def load_jsonl(path: Path):
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                raise ValueError(f"Linha vazia em {path}:{line_number}")
            row = json.loads(line)
            for field in ("sample_id", "text", "target_label"):
                if field not in row:
                    raise ValueError(
                        f"{path}:{line_number} sem campo {field!r}"
                    )
            rows.append(row)
    return rows

validation_rows = load_jsonl(VALIDATION_PATH)

if len(validation_rows) != 1350:
    raise ValueError(
        f"Validation possui {len(validation_rows)} registros; esperado 1350."
    )

counts = Counter(r["target_label"] for r in validation_rows)

if set(counts) != set(EXPECTED_LABELS):
    raise ValueError(f"Labels inesperados: {dict(counts)}")

if any(counts[label] != 150 for label in EXPECTED_LABELS):
    raise ValueError(
        f"Validation não possui 150 por classe: {dict(counts)}"
    )

VALIDATION_SHA256 = sha256_file(VALIDATION_PATH)

print("Validation records:", len(validation_rows))
print("SHA-256:", VALIDATION_SHA256)
print("Distribuição:", dict(counts))


Validation records: 1350
SHA-256: 6c48f802a9dfebc53463f565e5c619b8bcc155e0015361acb5e8cf7ad4143c7d
Distribuição: {'COBRADE_1_2_1_0_0': 150, 'COBRADE_1_2_2_0_0': 150, 'COBRADE_1_2_3_0_0': 150, 'COBRADE_1_4_1_3_1': 150, 'COBRADE_1_4_1_3_2': 150, 'COBRADE_2_3_1_1_0': 150, 'COBRADE_2_3_1_2_0': 150, 'INSUFFICIENT_INFO': 150, 'OTHER': 150}



## 1. Expressões suspeitas

Usamos a mesma ideia aplicada ao baseline TF-IDF.

Não removemos sinais COBRADE semanticamente legítimos como:

- `rio`, `margem`, `leito`;
- `correnteza`, `força`, `rapidamente`;
- `bueiro`, `drenagem`;
- `reserva`, `parque`;
- `fazenda`, `fumaça`, `qualidade do ar`;
- `fábrica`, `galpão`, `depósito`;
- `casas`, `moradias`, `residencial`.

O mascaramento se concentra em construções discursivas/administrativas
que apareceram associadas ao processo de geração sintética.


In [9]:

# 7. Cues suspeitos
SUSPICIOUS_CUES = {
    "other_generator_style": [
        r"\bcontinua sendo acompanhada\b",
        r"\bsendo acompanhada\b",
        r"\bequipes (?:estão )?avaliando (?:os )?danos\b",
        r"\bmoradores (?:estão )?evitando (?:o )?local\b",
        r"\bo acesso (?:foi )?interrompido temporariamente\b",
        r"\ba situação foi comunicada às equipes responsáveis\b",
        r"\ba situacao foi comunicada as equipes responsaveis\b",
        r"\bparte da região foi interditada\b",
        r"\bparte da regiao foi interditada\b",
        r"\bo trânsito está sendo desviado\b",
        r"\bo transito esta sendo desviado\b",
        r"\bo trânsito foi desviado\b",
        r"\bo transito foi desviado\b",
        r"\ba área foi isolada por segurança\b",
        r"\ba area foi isolada por seguranca\b",
        r"\brolou\b",
        r"\bdeu\b",
        r"\bteve\b",
        r"\bo pessoal viu\b",
    ],
    "insufficient_generator_style": [
        r"\bo registro preliminar menciona\b",
        r"\bo relato informa que\b",
        r"\ba ocorrência recebida descreve que\b",
        r"\ba ocorrencia recebida descreve que\b",
        r"\ba informação disponível indica que\b",
        r"\ba informacao disponivel indica que\b",
        r"\bo aviso recebido é incompleto\b",
        r"\bo aviso recebido e incompleto\b",
        r"\ba informação chegou sem contexto suficiente\b",
        r"\ba informacao chegou sem contexto suficiente\b",
        r"\bo que foi informado até o momento é apenas que\b",
        r"\bo que foi informado ate o momento e apenas que\b",
        r"\ba situação ainda está sendo verificada\b",
        r"\ba situacao ainda esta sendo verificada\b",
        r"\ba descrição recebida é incompleta\b",
        r"\ba descricao recebida e incompleta\b",
        r"\ba informação disponível ainda é muito vaga\b",
        r"\ba informacao disponivel ainda e muito vaga\b",
    ],
}

compiled_cues = []

for group, patterns in SUSPICIOUS_CUES.items():
    for pattern in patterns:
        compiled_cues.append(
            (group, re.compile(pattern, flags=re.IGNORECASE))
        )

def mask_suspicious_cues(text: str):
    masked = text
    matched_groups = []

    for group, pattern in compiled_cues:
        new_text, count = pattern.subn(" ", masked)

        if count:
            matched_groups.extend([group] * count)
            masked = new_text

    masked = re.sub(r"\s+", " ", masked).strip()
    masked = re.sub(r"\s+([,.;:!?])", r"\1", masked)

    if not masked:
        masked = "[sem_conteudo_apos_mascara]"

    return masked, matched_groups


In [10]:

# 8. Carregar tokenizer + modelo treinado
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_DIR
)

model.to(DEVICE)
model.eval()

id2label = {
    int(key): value
    for key, value in model.config.id2label.items()
}

print("Classes do modelo:")
for idx in sorted(id2label):
    print(idx, "->", id2label[idx])


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Classes do modelo:
0 -> COBRADE_1_2_1_0_0
1 -> COBRADE_1_2_2_0_0
2 -> COBRADE_1_2_3_0_0
3 -> COBRADE_1_4_1_3_1
4 -> COBRADE_1_4_1_3_2
5 -> COBRADE_2_3_1_1_0
6 -> COBRADE_2_3_1_2_0
7 -> OTHER
8 -> INSUFFICIENT_INFO


In [11]:

# 9. Inferência em lotes
MAX_LENGTH = 128
BATCH_SIZE = 64 if torch.cuda.is_available() else 16

@torch.no_grad()
def predict_texts(texts):
    all_pred_ids = []
    all_confidences = []
    all_probabilities = []

    for start in range(0, len(texts), BATCH_SIZE):
        batch = texts[start:start + BATCH_SIZE]

        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )

        encoded = {
            key: value.to(DEVICE)
            for key, value in encoded.items()
        }

        outputs = model(**encoded)
        probs = torch.softmax(outputs.logits, dim=-1)

        confidence, pred_ids = torch.max(probs, dim=-1)

        all_pred_ids.extend(pred_ids.cpu().tolist())
        all_confidences.extend(confidence.cpu().tolist())
        all_probabilities.extend(probs.cpu().numpy())

    return (
        np.asarray(all_pred_ids),
        np.asarray(all_confidences),
        np.asarray(all_probabilities),
    )


In [12]:

# 10. Preparar textos original + masked
original_texts = [r["text"] for r in validation_rows]
true_labels = [r["target_label"] for r in validation_rows]

masked_texts = []
cue_groups_per_row = []

cue_group_record_counts = Counter()
cue_group_match_counts = Counter()

for text in original_texts:
    masked, groups = mask_suspicious_cues(text)

    masked_texts.append(masked)
    cue_groups_per_row.append(groups)

    for group in set(groups):
        cue_group_record_counts[group] += 1

    cue_group_match_counts.update(groups)

affected_records = sum(bool(groups) for groups in cue_groups_per_row)

print("Validation records:", len(validation_rows))
print("Registros afetados:", affected_records)
print(
    "Share afetada:",
    round(affected_records / len(validation_rows), 4)
)
print("Por grupo:", dict(cue_group_record_counts))


Validation records: 1350
Registros afetados: 199
Share afetada: 0.1474
Por grupo: {'other_generator_style': 139, 'insufficient_generator_style': 60}


In [13]:

# 11. Rodar inferência original e masked
original_pred_ids, original_conf, original_probs = predict_texts(
    original_texts
)

masked_pred_ids, masked_conf, masked_probs = predict_texts(
    masked_texts
)

original_pred_labels = [
    id2label[int(i)]
    for i in original_pred_ids
]

masked_pred_labels = [
    id2label[int(i)]
    for i in masked_pred_ids
]

print("Inferência concluída.")


Inferência concluída.


In [14]:

# 12. Função de métricas
def compute_metrics(y_true, y_pred):
    return {
        "accuracy": float(
            accuracy_score(y_true, y_pred)
        ),
        "macro_precision": float(
            precision_score(
                y_true,
                y_pred,
                labels=EXPECTED_LABELS,
                average="macro",
                zero_division=0,
            )
        ),
        "macro_recall": float(
            recall_score(
                y_true,
                y_pred,
                labels=EXPECTED_LABELS,
                average="macro",
                zero_division=0,
            )
        ),
        "macro_f1": float(
            f1_score(
                y_true,
                y_pred,
                labels=EXPECTED_LABELS,
                average="macro",
                zero_division=0,
            )
        ),
    }

original_metrics = compute_metrics(
    true_labels,
    original_pred_labels,
)

masked_metrics = compute_metrics(
    true_labels,
    masked_pred_labels,
)

print("Original:")
print(json.dumps(original_metrics, indent=2))

print("\nMasked:")
print(json.dumps(masked_metrics, indent=2))


Original:
{
  "accuracy": 0.9985185185185185,
  "macro_precision": 0.9985283296541575,
  "macro_recall": 0.9985185185185185,
  "macro_f1": 0.9985185020574302
}

Masked:
{
  "accuracy": 0.98,
  "macro_precision": 0.9807694874278776,
  "macro_recall": 0.98,
  "macro_f1": 0.979359776744763
}


In [15]:

# 13. Métricas por classe
original_report = classification_report(
    true_labels,
    original_pred_labels,
    labels=EXPECTED_LABELS,
    output_dict=True,
    zero_division=0,
)

masked_report = classification_report(
    true_labels,
    masked_pred_labels,
    labels=EXPECTED_LABELS,
    output_dict=True,
    zero_division=0,
)

per_class_rows = []

for label in EXPECTED_LABELS:
    per_class_rows.append(
        {
            "target_label": label,
            "support": int(original_report[label]["support"]),
            "original_precision": float(original_report[label]["precision"]),
            "original_recall": float(original_report[label]["recall"]),
            "original_f1": float(original_report[label]["f1-score"]),
            "masked_precision": float(masked_report[label]["precision"]),
            "masked_recall": float(masked_report[label]["recall"]),
            "masked_f1": float(masked_report[label]["f1-score"]),
            "f1_delta": float(
                masked_report[label]["f1-score"]
                - original_report[label]["f1-score"]
            ),
        }
    )

per_class_df = pd.DataFrame(per_class_rows)
display(per_class_df)


,target_label,support,original_precision,original_recall,original_f1,masked_precision,masked_recall,masked_f1,f1_delta
0,COBRADE_1_2_1_0_0,150,1.000000,1.000000,1.000000,0.943396,1.000000,0.970874,-0.029126
1,COBRADE_1_2_2_0_0,150,1.000000,1.000000,1.000000,0.967742,1.000000,0.983607,-0.016393
2,COBRADE_1_2_3_0_0,150,0.993377,1.000000,0.996678,0.967742,1.000000,0.983607,-0.013071
3,COBRADE_1_4_1_3_1,150,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000
4,COBRADE_1_4_1_3_2,150,1.000000,0.993333,0.996656,1.000000,0.993333,0.996656,0.000000
5,COBRADE_2_3_1_1_0,150,0.993377,1.000000,0.996678,0.993377,1.000000,0.996678,0.000000
6,COBRADE_2_3_1_2_0,150,1.000000,1.000000,1.000000,0.993377,1.000000,0.996678,-0.003322
7,OTHER,150,1.000000,1.000000,1.000000,1.000000,0.833333,0.909091,-0.090909
8,INSUFFICIENT_INFO,150,1.000000,0.993333,0.996656,0.961290,0.993333,0.977049,-0.019606


In [16]:

# 14. Flips e confiança
rows_out = []
flip_rows = []

prediction_flips = 0
correct_to_wrong = 0
wrong_to_correct = 0
affected_confidence_deltas = []

for (
    source_row,
    original_text,
    masked_text,
    groups,
    pred_before,
    pred_after,
    conf_before,
    conf_after,
) in zip(
    validation_rows,
    original_texts,
    masked_texts,
    cue_groups_per_row,
    original_pred_labels,
    masked_pred_labels,
    original_conf,
    masked_conf,
):
    true_label = source_row["target_label"]

    before_correct = pred_before == true_label
    after_correct = pred_after == true_label
    flipped = pred_before != pred_after

    if groups:
        affected_confidence_deltas.append(
            float(conf_after - conf_before)
        )

    if flipped:
        prediction_flips += 1

    if before_correct and not after_correct:
        correct_to_wrong += 1

    if not before_correct and after_correct:
        wrong_to_correct += 1

    row = {
        "sample_id": source_row["sample_id"],
        "true_label": true_label,
        "original_prediction": pred_before,
        "masked_prediction": pred_after,
        "prediction_flipped": int(flipped),
        "original_correct": int(before_correct),
        "masked_correct": int(after_correct),
        "original_confidence": float(conf_before),
        "masked_confidence": float(conf_after),
        "confidence_delta": float(conf_after - conf_before),
        "cue_groups": "|".join(sorted(set(groups))),
        "cue_match_count": len(groups),
        "linguistic_profile": source_row.get("linguistic_profile"),
        "difficulty": source_row.get("difficulty"),
        "scenario_family": source_row.get("scenario_family"),
        "original_text": original_text,
        "masked_text": masked_text,
    }

    rows_out.append(row)

    if flipped:
        flip_rows.append(row)

print("Prediction flips :", prediction_flips)
print("Correct -> wrong :", correct_to_wrong)
print("Wrong -> correct :", wrong_to_correct)
print(
    "Mean confidence delta on affected:",
    float(np.mean(affected_confidence_deltas))
    if affected_confidence_deltas
    else 0.0,
)


Prediction flips : 25
Correct -> wrong : 25
Wrong -> correct : 0
Mean confidence delta on affected: -0.0515348640219051



## 2. Resultado principal


In [17]:

# 15. Resumo
print("=== BERTIMBAU SHORTCUT STRESS TEST ===")
print("Validation records :", len(validation_rows))
print("Registros afetados :", affected_records)
print("Prediction flips   :", prediction_flips)
print("Correct -> wrong   :", correct_to_wrong)
print("Wrong -> correct   :", wrong_to_correct)

print("\nOriginal:")
print(
    f"  accuracy : {original_metrics['accuracy']:.4f}"
)
print(
    f"  macro_f1 : {original_metrics['macro_f1']:.4f}"
)

print("\nMasked:")
print(
    f"  accuracy : {masked_metrics['accuracy']:.4f}"
)
print(
    f"  macro_f1 : {masked_metrics['macro_f1']:.4f}"
)

print("\nDelta:")
print(
    "  accuracy : "
    f"{masked_metrics['accuracy'] - original_metrics['accuracy']:+.4f}"
)
print(
    "  macro_f1 : "
    f"{masked_metrics['macro_f1'] - original_metrics['macro_f1']:+.4f}"
)

print("\nPor classe (F1 original -> masked):")
for row in per_class_rows:
    print(
        f"  {row['target_label']:<30} "
        f"{row['original_f1']:.4f} -> "
        f"{row['masked_f1']:.4f} "
        f"({row['f1_delta']:+.4f})"
    )


=== BERTIMBAU SHORTCUT STRESS TEST ===
Validation records : 1350
Registros afetados : 199
Prediction flips   : 25
Correct -> wrong   : 25
Wrong -> correct   : 0

Original:
  accuracy : 0.9985
  macro_f1 : 0.9985

Masked:
  accuracy : 0.9800
  macro_f1 : 0.9794

Delta:
  accuracy : -0.0185
  macro_f1 : -0.0192

Por classe (F1 original -> masked):
  COBRADE_1_2_1_0_0              1.0000 -> 0.9709 (-0.0291)
  COBRADE_1_2_2_0_0              1.0000 -> 0.9836 (-0.0164)
  COBRADE_1_2_3_0_0              0.9967 -> 0.9836 (-0.0131)
  COBRADE_1_4_1_3_1              1.0000 -> 1.0000 (+0.0000)
  COBRADE_1_4_1_3_2              0.9967 -> 0.9967 (+0.0000)
  COBRADE_2_3_1_1_0              0.9967 -> 0.9967 (+0.0000)
  COBRADE_2_3_1_2_0              1.0000 -> 0.9967 (-0.0033)
  OTHER                          1.0000 -> 0.9091 (-0.0909)
  INSUFFICIENT_INFO              0.9967 -> 0.9770 (-0.0196)


In [18]:

# 16. Mostrar todos os flips
flips_df = pd.DataFrame(flip_rows)

print("Total de flips:", len(flips_df))

if len(flips_df):
    display(
        flips_df[
            [
                "sample_id",
                "true_label",
                "original_prediction",
                "masked_prediction",
                "original_confidence",
                "masked_confidence",
                "cue_groups",
                "original_text",
                "masked_text",
            ]
        ]
    )
else:
    print("Nenhuma predição mudou.")


Total de flips: 25


,sample_id,true_label,original_prediction,masked_prediction,original_confidence,masked_confidence,cue_groups,original_text,masked_text
0,other_0236,OTHER,OTHER,INSUFFICIENT_INFO,0.995868,0.983526,other_generator_style,Há registro confirmado de a água de um poço fo...,Há registro confirmado de a água de um poço fo...
1,other_0319,OTHER,OTHER,COBRADE_1_2_3_0_0,0.995680,0.921269,other_generator_style,teve parte da encosta cedeu e a terra caiu sob...,parte da encosta cedeu e a terra caiu sobre a ...
2,other_0394,OTHER,OTHER,COBRADE_2_3_1_2_0,0.996216,0.436300,other_generator_style,rolou uma onda de calor mantem temperaturas mu...,uma onda de calor mantem temperaturas muito al...
3,other_0412,OTHER,OTHER,COBRADE_1_2_1_0_0,0.996358,0.719729,other_generator_style,deu um vendaval arrancou telhas e derrubou arv...,um vendaval arrancou telhas e derrubou arvores...
4,other_0429,OTHER,OTHER,COBRADE_1_2_2_0_0,0.996076,0.477418,other_generator_style,teve uma estrutura de concreto caiu dentro de ...,uma estrutura de concreto caiu dentro de uma o...
5,other_0456,OTHER,OTHER,COBRADE_1_2_2_0_0,0.996114,0.730518,other_generator_style,ta acontecendo uma barreira de terra desceu do...,ta acontecendo uma barreira de terra desceu do...
6,other_0460,OTHER,OTHER,COBRADE_1_2_1_0_0,0.996274,0.719729,other_generator_style,teve um vendaval arrancou telhas e derrubou ar...,um vendaval arrancou telhas e derrubou arvores...
7,other_0463,OTHER,OTHER,COBRADE_1_2_2_0_0,0.995677,0.575656,other_generator_style,o pessoal viu a estrutura de uma ponte caiu pa...,a estrutura de uma ponte caiu parcialmente e
8,other_0465,OTHER,OTHER,COBRADE_1_2_3_0_0,0.996175,0.921269,other_generator_style,rolou parte da encosta cedeu e a terra caiu so...,parte da encosta cedeu e a terra caiu sobre a ...
9,other_0478,OTHER,OTHER,INSUFFICIENT_INFO,0.995866,0.631135,other_generator_style,deu o calor intenso permanece por varios dias ...,o calor intenso permanece por varios dias sem ...


In [19]:

# 17. Casos com maior queda de confiança entre os afetados
predictions_df = pd.DataFrame(rows_out)

affected_df = predictions_df[
    predictions_df["cue_match_count"] > 0
].copy()

affected_df = affected_df.sort_values(
    "confidence_delta",
    ascending=True,
)

display(
    affected_df[
        [
            "sample_id",
            "true_label",
            "original_prediction",
            "masked_prediction",
            "original_confidence",
            "masked_confidence",
            "confidence_delta",
            "cue_groups",
            "original_text",
            "masked_text",
        ]
    ].head(30)
)


,sample_id,true_label,original_prediction,masked_prediction,original_confidence,masked_confidence,confidence_delta,cue_groups,original_text,masked_text
1298,other_0660,OTHER,OTHER,COBRADE_1_2_2_0_0,0.995956,0.424461,-0.571495,other_generator_style,vei o pessoal viu uma estrutura de concreto ca...,vei uma estrutura de concreto caiu dentro de u...
1291,other_0623,OTHER,OTHER,COBRADE_1_2_1_0_0,0.996241,0.425233,-0.571008,other_generator_style,aq deu a erosao costeira avancou e derrubou pa...,aq a erosao costeira avancou e derrubou parte ...
1255,other_0394,OTHER,OTHER,COBRADE_2_3_1_2_0,0.996216,0.436300,-0.559916,other_generator_style,rolou uma onda de calor mantem temperaturas mu...,uma onda de calor mantem temperaturas muito al...
1326,other_0860,OTHER,OTHER,COBRADE_1_2_1_0_0,0.996345,0.451068,-0.545278,other_generator_style,porra deu houve colapso de parte de uma ponte ...,porra houve colapso de parte de uma ponte sobr...
1261,other_0429,OTHER,OTHER,COBRADE_1_2_2_0_0,0.996076,0.477418,-0.518658,other_generator_style,teve uma estrutura de concreto caiu dentro de ...,uma estrutura de concreto caiu dentro de uma o...
1325,other_0859,OTHER,OTHER,OTHER,0.996215,0.486367,-0.509848,other_generator_style,caralho rolou a estiagem esta secando pocos e ...,caralho a estiagem esta secando pocos e pequen...
1293,other_0631,OTHER,OTHER,OTHER,0.996304,0.490520,-0.505784,other_generator_style,aq rolou o terreno perdeu grande parte do solo...,aq o terreno perdeu grande parte do solo por e...
1282,other_0555,OTHER,OTHER,INSUFFICIENT_INFO,0.995827,0.514874,-0.480953,other_generator_style,rolou as temperaturas seguem mt acima do norma...,as temperaturas seguem mt acima do normal dura...
1308,other_0759,OTHER,OTHER,INSUFFICIENT_INFO,0.996347,0.538408,-0.457939,other_generator_style,teve o mar esta retirando areia e destruindo a...,o mar esta retirando areia e destruindo a faix...
1319,other_0819,OTHER,OTHER,COBRADE_1_2_2_0_0,0.996208,0.575657,-0.420551,other_generator_style,rolou a estrutura de uma ponte caiu parcialmen...,a estrutura de uma ponte caiu parcialmente e



## 3. Exportar o diagnóstico

Os resultados serão salvos na mesma pasta lógica do experimento:

```text
ml/evaluation/nlp/kamara_bertimbau_v1/
```

Nenhum arquivo do modelo é alterado.


In [20]:

# 18. Salvar resultados
EXPORT_ROOT = Path("/content/kamara_bertimbau_shortcut_export")
EVAL_DIR = (
    EXPORT_ROOT
    / "ml"
    / "evaluation"
    / "nlp"
    / "kamara_bertimbau_v1"
)

EVAL_DIR.mkdir(parents=True, exist_ok=True)

per_class_df.to_csv(
    EVAL_DIR / "shortcut_stress_per_class_validation.csv",
    index=False,
)

predictions_df.to_csv(
    EVAL_DIR / "shortcut_stress_predictions_validation.csv",
    index=False,
)

pd.DataFrame(flip_rows).to_csv(
    EVAL_DIR / "shortcut_stress_flips_validation.csv",
    index=False,
)

report = {
    "diagnostic": "bertimbau_synthetic_shortcut_masking_validation_v1",
    "model": "kamara_bertimbau_v1",
    "validation_records": len(validation_rows),
    "validation_sha256": VALIDATION_SHA256,
    "test_used": False,
    "affected_records": affected_records,
    "affected_share": affected_records / len(validation_rows),
    "prediction_flips": prediction_flips,
    "correct_to_wrong": correct_to_wrong,
    "wrong_to_correct": wrong_to_correct,
    "mean_confidence_delta_on_affected": (
        float(np.mean(affected_confidence_deltas))
        if affected_confidence_deltas
        else 0.0
    ),
    "cue_group_record_counts": dict(cue_group_record_counts),
    "cue_group_match_counts": dict(cue_group_match_counts),
    "original_metrics": original_metrics,
    "masked_metrics": masked_metrics,
    "metric_delta": {
        key: masked_metrics[key] - original_metrics[key]
        for key in original_metrics
    },
    "per_class": {
        row["target_label"]: row
        for row in per_class_rows
    },
    "interpretation_note": (
        "A queda observada mede dependência das expressões mascaradas, "
        "não ausência de todo viés sintético."
    ),
}

with (
    EVAL_DIR / "shortcut_stress_test_validation.json"
).open("w", encoding="utf-8") as f:
    json.dump(
        report,
        f,
        ensure_ascii=False,
        indent=2,
    )
    f.write("\n")

print("Resultados salvos em:", EVAL_DIR)


Resultados salvos em: /content/kamara_bertimbau_shortcut_export/ml/evaluation/nlp/kamara_bertimbau_v1


In [21]:

# 19. Criar ZIP
ZIP_BASE = Path("/content/kamara_bertimbau_shortcut_stress")

zip_path = shutil.make_archive(
    str(ZIP_BASE),
    "zip",
    root_dir=EXPORT_ROOT,
)

print("ZIP:", zip_path)
print(
    "Tamanho (KB):",
    round(Path(zip_path).stat().st_size / 1024, 2),
)


ZIP: /content/kamara_bertimbau_shortcut_stress.zip
Tamanho (KB): 70.22


In [22]:

# 20. Download
from google.colab import files

files.download(
    "/content/kamara_bertimbau_shortcut_stress.zip"
)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


# 4. Próxima decisão

Depois deste notebook:

- se a queda de Macro-F1 for pequena, congelamos `kamara_bertimbau_v1`;
- se a queda for grande, revisamos principalmente `OTHER` e `INSUFFICIENT_INFO`;
- só depois fazemos **uma avaliação única** no `test.jsonl` sintético;
- a avaliação realmente importante continuará sendo o conjunto
  manual/externo independente.

Baixe também este notebook executado e salve localmente como:

```text
ml/notebooks/nlp/02_stress_test_bertimbau_shortcuts.ipynb
```
